# Lines of code: library and hand-written answers

How big are the answers that use a library, next to the ones written by hand? This notebook compares them for each model, reward and prompt, on LibReuseBench test (216 tasks, 5 answers each) at each run's final epoch (epoch 4), as in `paper_figures.ipynb`.

The two prompts are the task as it is (**plain**), and the task with the suffix *"Use an external library when appropriate."* (**+ suffix**).

**How the values are computed:**

- every value is averaged within each task's 5 answers, then over tasks (as in table 1);
- **library share** is the share of answers that import and use a third-party library, counting every answer;
- **lines** are logical lines of code (radon's `lloc`: one per statement, so blank lines, comments and statements split over several lines don't change it), and **MI** is radon's maintainability index (0-100, higher is easier to maintain). Both count only answers with code radon can measure;
- **lines, hand** and **lines, library** each average over the tasks with at least one answer of that kind, so they mostly cover different tasks. The difference in brackets is therefore not a saving on the same task;
- **lines saved** is the like-for-like measure, from `paper_figures.ipynb`: an answer that uses a library saves however many lines shorter it is than the same model's base version's average hand-written answer to the task (plain prompt), and an answer without a library saves 0. Tasks the base model never answered by hand are left out;
- **tokens** are the tokens generated per answer, counting every answer, cut-off ones included. They are split into **reasoning** (up to the closing `</think>`) and **response** (the final answer after it). An answer cut off before `</think>` is all reasoning.

**The token split is an estimate.** The saved records hold only each answer's total token count, so the total is split in proportion to the characters in the reasoning and in the response. Code packs more tokens per character than prose does, so the response tokens come out somewhat low and the reasoning tokens somewhat high. The total is exact.

In [1]:
"""Compare the size of library and hand-written answers on LibReuseBench test, per model, reward and prompt."""

import os
from functools import cache
from pathlib import Path

import pandas as pd
from IPython.display import display
from llm_cgr import load_jsonl
from lure import split_reasoning

from src.utils.paths import baseline_dir, training_run_dir


# run from the repo root (the folder holding .git), so the relative paths src uses
# (config/, output/) resolve - its results are in output/
REPO_ROOT = next(
    folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / ".git").exists()
)
os.chdir(REPO_ROOT)

MODELS = {
    "qwen3-8b": "Qwen3-8B",
    "olmo3-7b": "OLMo-3-7B-Think",
    "seed-coder-8b": "Seed-Coder-8B-Reasoning",
}
REWARDS = ["import", "usage", "target"]
# the final training runs, one per reward, as in notebooks/paper_figures.ipynb
RUNS: dict[str, dict[str, str]] = {
    "qwen3-8b": {
        "import": "qwen3-8b-f2ca",
        "usage": "qwen3-8b-3079",
        "target": "qwen3-8b-f57b",
    },
    "olmo3-7b": {
        "import": "olmo3-7b-e93f",
        "usage": "olmo3-7b-cb85",
        "target": "olmo3-7b-15e4",
    },
    "seed-coder-8b": {
        "import": "seed-coder-8b-50e9",
        "usage": "seed-coder-8b-eee7",
        "target": "seed-coder-8b-5ad0",
    },
}
FINAL_EPOCH = 4
DATASET = "libreusebench_test"
DECODING_PROFILE = "default_16k"
# the prompt styles, as the result files name them, with the name the table uses
PROMPT_STYLES = {
    "base": "plain",
    "appropriate": "+ suffix",
}

## Load the answers

Each eval's answers come from its saved records (`*.generations.jsonl`), one row per answer. Each answer's token count is split into reasoning and response as it is loaded.

In [2]:
def results_file(
    model: str,
    reward: str | None,
    style: str,
) -> Path:
    """
    Find the saved records of one eval: the base model's (no reward), or one run's final
    checkpoint.

    Returns the file's path.
    """
    if reward is None:
        folder = baseline_dir(model)
    else:
        run_dir = training_run_dir(
            base_model=model,
            run_id=RUNS[model][reward],
        )
        folder = run_dir / f"epoch{FINAL_EPOCH}"
    return folder / f"{DATASET}-{DECODING_PROFILE}-{style}.generations.jsonl"


def load_records(
    model: str,
    reward: str | None,
    style: str,
) -> pd.DataFrame:
    """
    Load one eval's answers, and split each answer's tokens into reasoning and response
    (see the note at the top).

    Returns one row per answer.
    """
    path = results_file(
        model=model,
        reward=reward,
        style=style,
    )
    records = pd.DataFrame(load_jsonl(file_path=str(path)))

    # the reasoning is the text before </think>, the response the text after it
    parts = records["response"].map(split_reasoning)
    reasoning_characters = parts.str[0].str.len()
    response_characters = parts.str[1].str.len()
    # shortcut: tokens are split by character share, as the records hold only the total.
    # count each part with the model's own tokenizer if exact values are needed
    reasoning_share = reasoning_characters / (
        reasoning_characters + response_characters
    )
    records["reasoning_tokens"] = records["completion_tokens"] * reasoning_share
    records["response_tokens"] = (
        records["completion_tokens"] - records["reasoning_tokens"]
    )
    return records


@cache
def hand_written_lines(
    model: str,
) -> pd.Series:
    """
    A base model's average hand-written answer (no library, plain prompt) to each task, in
    logical lines of code. Lines saved compares a library-using answer with it.

    Returns the lines of code per task, for the tasks the base model answered by hand.
    """
    records = load_records(
        model=model,
        reward=None,
        style="base",
    )
    by_hand = records[records["lloc"].notna() & ~records["external_lib_used"]]
    return by_hand.groupby("task_id")["lloc"].mean()

## The table

One row per model, version (the base model, or the run trained with each reward) and prompt.

In [3]:
def task_average(
    values: pd.Series,
    records: pd.DataFrame,
) -> float:
    """
    Average a value within each task, then over tasks. Answers with no value (NaN) are
    left out, and so are tasks with none.

    Returns the average.
    """
    return values.groupby(records["task_id"]).mean().mean()


def summarise(
    model: str,
    reward: str | None,
    style: str,
) -> dict:
    """
    Work out every value in one row of the table (see the notes at the top).

    Returns the row.
    """
    records = load_records(
        model=model,
        reward=reward,
        style=style,
    )

    library = records["external_lib_used"]
    measured = records["lloc"].notna()
    lines = records["lloc"].where(measured)

    # lines saved: shorter than the base model's hand-written answer if it uses a
    # library, else 0
    reference = records["task_id"].map(hand_written_lines(model=model))
    saved = (reference - lines).where(library, 0.0)
    saved = saved.where(measured & reference.notna())

    return {
        "Model": MODELS[model],
        "Version": reward or "base",
        "Prompt": PROMPT_STYLES[style],
        "Library share": task_average(
            values=library.astype(float),
            records=records,
        ),
        "Avg lines": task_average(
            values=lines,
            records=records,
        ),
        "Lines, hand": task_average(
            values=lines.where(~library),
            records=records,
        ),
        "Lines, library": task_average(
            values=lines.where(library),
            records=records,
        ),
        "Avg MI": task_average(
            values=records["maintainability_index"].where(measured),
            records=records,
        ),
        "Lines saved": task_average(
            values=saved,
            records=records,
        ),
        "Tokens": task_average(
            values=records["completion_tokens"].astype(float),
            records=records,
        ),
        "Reasoning tokens": task_average(
            values=records["reasoning_tokens"],
            records=records,
        ),
        "Response tokens": task_average(
            values=records["response_tokens"],
            records=records,
        ),
    }


table = pd.DataFrame(
    [
        summarise(
            model=model,
            reward=reward,
            style=style,
        )
        for model in MODELS
        for reward in [None, *REWARDS]
        for style in PROMPT_STYLES
    ]
)


def with_difference(
    row: pd.Series,
) -> str:
    """
    Write a row's library lines with the difference from its hand-written lines.

    Returns text such as "17.3 (−10.1)".
    """
    difference = row["Lines, library"] - row["Lines, hand"]
    return f"{row['Lines, library']:.1f} ({difference:+.1f})".replace("-", "−")


# the table as it is shown: the model named once per group, the difference in brackets
shown = table.copy()
shown["Lines, library"] = table.apply(with_difference, axis=1)
shown["Model"] = shown["Model"].where(shown["Model"] != shown["Model"].shift(), "")
display(
    shown.style.hide(axis="index").format(
        {
            "Library share": "{:.1%}",
            "Avg lines": "{:.1f}",
            "Lines, hand": "{:.1f}",
            "Avg MI": "{:.1f}",
            "Lines saved": "{:.1f}",
            "Tokens": "{:,.0f}",
            "Reasoning tokens": "{:,.0f}",
            "Response tokens": "{:,.0f}",
        }
    )
)

Model,Version,Prompt,Library share,Avg lines,"Lines, hand","Lines, library",Avg MI,Lines saved,Tokens,Reasoning tokens,Response tokens
Qwen3-8B,base,plain,15.7%,25.7,27.4,17.3 (−10.0),76.2,0.7,"9,158","8,238",919
,base,+ suffix,46.2%,21.0,24.2,17.2 (−7.0),82.8,5.1,"8,244","7,377",866
,import,plain,37.7%,22.7,26.2,18.3 (−7.9),81.0,3.9,"7,434","6,566",868
,import,+ suffix,72.5%,18.8,23.8,16.7 (−7.1),86.4,7.9,"6,678","5,835",843
,usage,plain,34.6%,22.8,25.9,17.7 (−8.2),80.0,3.5,"7,146","6,240",906
,usage,+ suffix,69.3%,18.8,23.6,16.8 (−6.8),85.2,7.9,"6,462","5,581",882
,target,plain,41.8%,22.2,26.1,17.6 (−8.5),80.3,4.5,"6,868","5,992",876
,target,+ suffix,75.2%,18.0,23.0,16.3 (−6.7),85.5,8.5,"6,190","5,344",846
OLMo-3-7B-Think,base,plain,2.2%,31.6,32.5,26.8 (−5.7),68.4,0.6,"11,283","10,666",617
,base,+ suffix,43.5%,23.7,28.7,19.2 (−9.5),76.2,8.0,"9,857","9,209",647
